# 01 — Tu primer DataFrame

Acá vas a:
1. Crear una `SparkSession`.
2. Crear un DataFrame a partir de una lista de Python.
3. Aplicar transforms y una action.
4. Leer la Spark UI para ver lo que pasó.

**Objetivo**: entender la diferencia entre *definir* un plan y *ejecutarlo*.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = (
    SparkSession.builder
    .appName("01_primer_dataframe")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("App ID:", spark.sparkContext.applicationId)

## 1. DataFrame desde una lista

Tres formas equivalentes. La tercera es la más explícita.

In [ ]:
# Forma 1: inferencia de schema (no usar en prod, lento y frágil)
data = [
    ("Ana", 28, "Bogotá"),
    ("Luis", 35, "Medellín"),
    ("Sofía", 22, "Bogotá"),
    ("Pedro", 41, "Cali"),
    ("Camila", 29, "Bogotá"),
    ("Andrés", 33, "Medellín"),
]
df = spark.createDataFrame(data, ["nombre", "edad", "ciudad"])
df.printSchema()
df.show()

In [ ]:
# Forma 2: schema explícito (recomendado en prod)
schema = StructType([
    StructField("nombre", StringType(), False),
    StructField("edad", IntegerType(), False),
    StructField("ciudad", StringType(), False),
])
df = spark.createDataFrame(data, schema=schema)
df.printSchema()

## 2. Transforms vs Actions

Fijate cómo **nada se ejecuta** hasta el `show()`.

In [ ]:
mayores = df.filter(F.col("edad") >= 30)        # transform (lazy)
bogota = mayores.filter(F.col("ciudad") == "Bogotá")  # transform (lazy)
seleccion = bogota.select("nombre", "edad")    # transform (lazy)

print("Tipo de seleccion:", type(seleccion))    # es un DataFrame, no un valor
print("Acciones todavía no se ejecutaron. No se imprimió nada.")

In [ ]:
# Ahora sí: action
seleccion.show()
print("Cantidad de filas:", seleccion.count())  # otra action

## 3. Mirar la Spark UI

Mientras estas celdas corrieron, abrí <http://localhost:4040>. Vas a ver:

- **Jobs**: 3 jobs (uno por cada action: `show` de schema, `show` de seleccion, `count`).
- **SQL tab**: el plan optimizado de cada job. Fijate como el `filter` se aplicó **antes** que el `select` — eso lo hizo Catalyst, no vos.
- **DAG**: la visualización del flujo de transforms.

Si la UI dice *No jobs found*, refrescá después de cada celda. Cuando el
driver está idle, la UI muestra estado vacío.

## 4. Columnas derivadas y agregaciones

In [ ]:
df_con_categoria = df.withColumn(
    "rango_etario",
    F.when(F.col("edad") < 30, "joven")
     .when(F.col("edad") < 40, "adulto")
     .otherwise("senior"),
)
df_con_categoria.show()

In [ ]:
df.groupBy("ciudad").agg(
    F.count("*").alias("n"),
    F.round(F.avg("edad"), 1).alias("edad_promedio"),
    F.min("edad").alias("min_edad"),
    F.max("edad").alias("max_edad"),
).orderBy(F.col("n").desc()).show()

## ✅ Checkpoint

Si llegaste hasta acá y todo corrió, ya entendés:

- [x] Cómo crear un DataFrame con schema explícito.
- [x] La diferencia entre transform y action.
- [x] Que Catalyst reordena operaciones para optimizar.
- [x] Dónde mirar para diagnosticar (`localhost:4040`).

**Siguiente**: `02_lectura_archivos.ipynb` — leemos CSVs, Parquet, y JSON de verdad.